# [M3] XÂY MODEL AI SÂU HƠN BẰNG PYTORCH
## Buổi S05: Chống Overfitting, Learning Curves & Model Checkpoint

---

### Metadata Khóa Học
* **Module:** M3 — XÂY MODEL AI SÂU HƠN BẰNG PYTORCH
* **Buổi:** S05
* **Thời lượng dự kiến:** 120 phút
* **Sản phẩm đầu ra (Session Output):** Notebook PyTorch có training loop + Checkpoint (.pt)
* **Mức độ liên quan VOAI / Olympic AI:** Bảo tồn mô hình tối ưu, ngăn ngừa rớt hạng do học vẹt trong phòng thi.

#### Mục tiêu học tập (Learning Objectives)
- Chẩn đoán chính xác hiện tượng Quá khớp (Overfitting) thông qua Đồ thị đường cong học tập (Train vs Validation Loss Curves).
- Áp dụng thành thạo hai kỹ thuật điều quy (Regularization) kinh điển: Suy giảm trọng số (Weight Decay / L2) và Dropout (`nn.Dropout`).
- Làm chủ kỹ thuật Dừng sớm (Early Stopping) để tự động bảo tồn trạng thái mô hình tốt nhất.
- Thực hành Lưu trữ (Save) và Tái nạp (Load) Model Checkpoint (`state_dict`) đạt chuẩn công nghiệp và hoàn thành sản phẩm Module 3.

#### Điều kiện tiên quyết (Prerequisites)
- Hoàn thành Buổi M3_S04 (Đã viết thành thạo Training & Validation Loop).
- Hiểu khái niệm Overfitting từ Module M2.

---


## 1. WHY? — TẠI SAO PHẢI HỌC CHỦ ĐỀ NÀY?


Trong các kỳ thi AI kéo dài nhiều giờ, tài nguyên tính toán luôn có hạn. Nếu bạn để mô hình chạy 200 epochs mà không kiểm soát, có thể sau epoch thứ 20 mô hình đã bắt đầu "học vẹt" (Overfitting), khiến hiệu năng trên tập kiểm tra thực tế tụt dốc không phanh. Chưa kể, nếu giữa chừng xảy ra sự cố sập nguồn, mất kết nối Colab hoặc hết thời gian thi, toàn bộ trọng số học được sẽ biến mất vĩnh viễn!

Làm chủ kỹ thuật **Điều quy hóa (Regularization)**, đọc **Learning Curves**, và cơ chế **Lưu vết Checkpoint (Save/Load State Dict)** chính là áo giáp bảo vệ thành quả của bạn: Nó đảm bảo bạn luôn giữ lại được phiên bản mô hình thông minh nhất và sẵn sàng phục hồi trạng thái chỉ trong 1 giây để nộp bài thi!



## 2. CONCEPT — NỀN TẢNG LÝ THUYẾT & TRỰC GIÁC


### 1. Đọc vị Đường cong học tập (Learning Curves)
* **Underfitting (Chưa khớp):** Cả Train Loss và Val Loss đều cao và không chịu giảm. Nguyên nhân: Mô hình quá yếu hoặc chưa học đủ lâu.
* **Good Fit (Hội tụ tối ưu):** Cả Train Loss và Val Loss cùng giảm đều và tiệm cận một giá trị thấp ổn định.
* **Overfitting (Quá khớp):** Train Loss tiếp tục giảm sâu, nhưng **Val Loss bắt đầu quay đầu tăng ngược trở lại**! Đây là thời điểm vàng cần dừng huấn luyện ngay lập tức.

### 2. Các kỹ thuật chống Overfitting trong PyTorch
1. **Weight Decay (L2 Regularization):** Phạt các trọng số có độ lớn quá cao, ép mô hình chọn các nghiệm êm dịu hơn:
   `optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-4)`
2. **Dropout (`nn.Dropout(p=0.3)`):** Trong mỗi lượt huấn luyện, ngẫu nhiên "tắt" (gán bằng 0) một tỷ lệ $p$ các nơ-ron, buộc mạng không thể phụ thuộc vào một cá nhân nơ-ron nào mà phải học các đặc trưng tập thể mạnh mẽ.

### 3. Cơ chế Lưu và Nạp Model Checkpoint chuẩn (`state_dict`)
Trong PyTorch, chuẩn mực vàng là **chỉ lưu bảng tham số `state_dict`** (chứa tensor trọng số và bias), không lưu toàn bộ đối tượng class để tránh lỗi không tương thích môi trường:
* **Lưu Checkpoint:**
  ```python
  torch.save({
      'epoch': epoch,
      'model_state_dict': model.state_dict(),
      'optimizer_state_dict': optimizer.state_dict(),
      'val_loss': best_val_loss,
  }, 'best_checkpoint.pt')
  ```
* **Nạp lại mô hình:**
  ```python
  model = MyModel()
  checkpoint = torch.load('best_checkpoint.pt')
  model.load_state_dict(checkpoint['model_state_dict'])
  model.eval() # Bắt buộc chuyển sang eval sau khi load!
  ```



## 3. VISUAL INTUITION — TRỰC QUAN HÓA KIẾN THỨC


```
Trực quan hóa Overfitting & Early Stopping:
  Loss
   ▲
   │        \                     / ──> Validation Loss (Bắt đầu tăng: Overfitting!)
   │         \     VÙNG TỐI ƯU   /
   │          \         │       /
   │           \        ▼      /
   │            \───────★─────'  <─── BEST CHECKPOINT (Lưu tại đây!)
   │                │              \───────────────────> Train Loss (Tiếp tục giảm sâu: Học vẹt)
   └───────────────────────────────────► Epochs
```



## 4. CODE IMPLEMENTATION & 5. WALKTHROUGH

*Toàn bộ code dưới đây được thiết kế thực tế, có thể chạy độc lập không phụ thuộc tải dữ liệu bên ngoài.*

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import os

# 1. Thiet ke mang MLP co Regularization manh me: Dropout
class RegularizedMLP(nn.Module):
    def __init__(self, input_dim=10, hidden_dim=32, dropout_rate=0.3):
        super(RegularizedMLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate), # Ngau nhien tat 30% no-ron khi train de chong Overfitting
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(hidden_dim // 2, 1)
        )
        
    def forward(self, x):
        return self.net(x)

# Khoi tao mo hinh
model = RegularizedMLP(input_dim=10, hidden_dim=32, dropout_rate=0.3)
print("===== MO HINH REGULARIZED MLP CO DROPOUT =====")
print(model)

### 🔍 Code Walkthrough (Khối 1)
Khối code tích hợp kỹ thuật điều quy Dropout trực tiếp vào kiến trúc:
- `nn.Dropout(p=0.3)` được chèn xen kẽ giữa các tầng ẩn.
- Giúp phá vỡ sự đồng thích nghi (Co-adaptation) của các nơ-ron, giảm thiểu triệt để nguy cơ quá khớp.


In [ ]:
# 2. Huan luyen, Luu Checkpoint tai diem Val Loss tot nhat va Tai nap lai
torch.manual_seed(42)
X_dummy = torch.randn(200, 10)
y_dummy = torch.randint(0, 2, (200, 1)).float()

criterion = nn.BCEWithLogitsLoss()
# Tich hop L2 Regularization (weight_decay=1e-4) trong Adam
optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-4)

best_val_loss = float('inf')
checkpoint_path = 'best_model_checkpoint.pt'

# Gia lap qua trinh huan luyen va luu checkpoint
epochs = 15
train_history = []
val_history = []

for epoch in range(1, epochs + 1):
    model.train()
    optimizer.zero_grad()
    out = model(X_dummy[:150])
    loss = criterion(out, y_dummy[:150])
    loss.backward()
    optimizer.step()
    
    # Validation step
    model.eval()
    with torch.no_grad():
        val_out = model(X_dummy[150:])
        v_loss = criterion(val_out, y_dummy[150:]).item()
        
    train_history.append(loss.item())
    val_history.append(v_loss)
    
    # Kiem tra va Luu Checkpoint neu dat Val Loss ky luc moi
    if v_loss < best_val_loss:
        best_val_loss = v_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_loss': best_val_loss
        }, checkpoint_path)

print(f"-> Da luu Checkpoint tot nhat tai Val Loss = {best_val_loss:.4f} vao file: {checkpoint_path}")

# 3. Kiem thu Tai nap lai (Load Checkpoint) de kiem tra tinh toan ven
new_model = RegularizedMLP(input_dim=10, hidden_dim=32)
loaded_checkpoint = torch.load(checkpoint_path, weights_only=True)
new_model.load_state_dict(loaded_checkpoint['model_state_dict'])
new_model.eval() # Chuyen sang che do danh gia

print("-> Tai nap Checkpoint thanh cong! San sang suy dien (Inference Ready).")
print(f"Thong tin Checkpoint luu o Epoch: {loaded_checkpoint['epoch']}")

### 🔍 Code Walkthrough (Khối 2)
Khối code minh họa chu trình chuẩn mực về Checkpointing trong Deep Learning:
- Theo dõi `val_loss` qua từng epoch và tự động ghi đè file `best_model_checkpoint.pt` khi đạt mốc kỷ lục mới.
- Lưu trữ đầy đủ siêu dữ liệu: `model_state_dict`, `optimizer_state_dict`, số epoch.
- Tái nạp mô hình vào một đối tượng mới hoàn toàn bằng `load_state_dict` và chuyển ngay sang `eval()`.


## 6. PRACTICE — BÀI TẬP THỰC HÀNH


### Bài 1 (Easy): Vẽ biểu đồ Learning Curves
Sử dụng dữ liệu `train_history` và `val_history` ở trên, dùng Matplotlib vẽ đồ thị đường cong học tập với 2 đường: Train Loss (màu xanh) và Val Loss (màu đỏ). Đánh dấu điểm Epoch có Val Loss nhỏ nhất bằng một ngôi sao trên biểu đồ.

### Bài 2 (Easy): Thử nghiệm thay đổi tỷ lệ Dropout
Thay đổi `dropout_rate` thành $0.7$ (quá cao) và $0.05$ (quá thấp). Quan sát sự chênh lệch giữa tốc độ học của tập Train và tập Validation.

### Bài 3 (Medium): Xây dựng Lớp EarlyStopping tùy biến
Viết một class `EarlyStopping(patience=5, min_delta=0.001)` có phương thức theo dõi `step(val_loss)`. Nếu sau 5 epoch liên tiếp mà `val_loss` không giảm thêm ít nhất `0.001`, tự động kích hoạt cờ dừng `should_stop = True` để ngắt vòng lặp huấn luyện.

### Bài 4 (Medium): Nạp Checkpoint để tiếp tục huấn luyện (Resume Training)
Giả sử máy tính bị mất điện ở epoch thứ 15. Hãy viết đoạn code nạp lại cả `model_state_dict` VÀ `optimizer_state_dict` từ checkpoint đã lưu, sau đó tiếp tục huấn luyện từ epoch 16 đến epoch 30 một cách liên tục không đứt đoạn.

### Bài 5 (Challenge): Xuất dự đoán ra file `submission.csv` từ Checkpoint
Viết một hàm `generate_submission(model, test_loader, output_file='test_predictions.csv')` nạp trọng số từ file checkpoint tốt nhất, chạy dự đoán trên tập dữ liệu kiểm tra, và xuất ra file CSV gồm 2 cột: `ID` và `Probability_Prediction` chuẩn quy cách phòng thi.



## 7. VOAI CONNECTION — CHIẾN LƯỢC PHÒNG THI OLYMPIC


Trong thi đấu Olympic AI, quy chế thi thường yêu cầu: *"Thí sinh phải nộp cả file Notebook VÀ file trọng số mô hình đã huấn luyện (.pt hoặc .pth)"*. Nếu notebook của bạn huấn luyện mất 3 tiếng nhưng bạn không lưu Checkpoint, khi ban giám khảo chạy lại để chấm điểm và gặp sự cố timeout, bài thi của bạn sẽ bị đánh rớt. Luôn lưu lại `best_checkpoint.pt` sau mỗi epoch là thói quen sống còn của một tuyển thủ chuyên nghiệp.



## 8. DEBUGGING CHALLENGE — BẮT LỖI THỰC CHIẾN


**Tình huống thực tế:** Một bạn học viên nạp lại mô hình từ file checkpoint nhưng quên gọi lệnh `model.eval()`, khiến kết quả dự đoán trên cùng một mẫu thử mỗi lần chạy lại cho ra một con số khác nhau:

```python
# CODE CÓ LỖI BẤT ỔN ĐỊNH DO QUÊN MODEL.EVAL():
# Khi quen model.eval(), tang nn.Dropout van tiep tuc ngau nhien tat 30% no-ron
# moi lan du doan, lam ket qua submission bi nhieu va giam diem dang tiec!
```

**Nhiệm vụ của bạn:**
1. Giải thích tại sao Dropout chỉ được phép hoạt động trong lúc huấn luyện và bắt buộc phải tắt khi đánh giá.
2. Viết lại đoạn code nạp checkpoint bảo đảm luôn có lệnh `model.eval()` ngay lập tức.



## 9. KNOWLEDGE CHECK — KIỂM TRA ĐỘ HIỂU BẢN CHẤT


1. Khi quan sát đồ thị Learning Curves, dấu hiệu rõ ràng nhất cho thấy mô hình đang bị Quá khớp (Overfitting) là gì?
   - A. Cả Train Loss và Val Loss đều không thay đổi qua các epoch.
   - B. Train Loss tiếp tục giảm sâu, nhưng Validation Loss bắt đầu tăng ngược trở lại.
   - C. Train Loss tăng cao hơn Validation Loss.
   - D. Tốc độ quạt tản nhiệt của máy tính kêu to hơn.

2. Tầng `nn.Dropout(p=0.4)` trong PyTorch thực hiện thao tác gì trong quá trình huấn luyện?
   - A. Xóa bỏ vĩnh viễn 40% số nơ-ron của tầng đó.
   - B. Trong mỗi Mini-batch, ngẫu nhiên đặt giá trị đầu ra của 40% số nơ-ron thành số 0 để tránh việc các nơ-ron quá phụ thuộc vào nhau.
   - C. Giảm tốc độ học xuống 40%.
   - D. Giảm kích thước batch size đi 40%.

3. Phương thức chuẩn mực và an toàn nhất được khuyến nghị bởi đội ngũ phát triển PyTorch để lưu trạng thái mô hình là gì?
   - A. `torch.save(model, 'model.pt')` (Lưu toàn bộ đối tượng class).
   - B. `torch.save(model.state_dict(), 'model_weights.pt')` (Chỉ lưu từ điển trọng số và tham số).
   - C. Xuất ra file Excel.
   - D. Lưu code Python ra file `.txt`.

4. Cơ chế "Early Stopping" (Dừng sớm) trong huấn luyện mạng nơ-ron có mục đích chính là gì?
   - A. Để máy tính đỡ tốn điện.
   - B. Tự động ngắt vòng lặp huấn luyện khi mô hình không còn cải thiện hiệu năng trên tập Validation sau một số epoch quy định (Patience), tránh lãng phí thời gian và ngăn chặn Overfitting.
   - C. Tự động xóa các dữ liệu ngoại lai.
   - D. Tăng độ sâu của mạng nơ-ron.

5. Sản phẩm đầu ra chính thức mà học viên bắt buộc phải hoàn thành khi kết thúc Module M3 là gì?
   - A. Một bài thuyết trình PowerPoint về lịch sử AI.
   - B. Một Notebook PyTorch hoàn chỉnh có Training Loop tự viết và file Model Checkpoint (.pt) đã được lưu thành công.
   - C. Một website thương mại điện tử.
   - D. Một bài viết blog dài 2000 từ.



## 10. SESSION SUMMARY — TỔNG KẾT BUỔI HỌC


* **What I learned:**
  - Kỹ năng chẩn đoán Overfitting qua đường cong học tập Train/Val Loss.
  - Cách ứng dụng Regularization: Weight Decay (L2) và Dropout (`nn.Dropout`).
  - Cơ chế Early Stopping và kỹ thuật lưu trữ/tái nạp Checkpoint chuẩn mực (`state_dict`).
* **What I can now do:**
  - Tự tin kiểm soát quá trình huấn luyện của mạng nơ-ron sâu, không để mô hình bị học vẹt.
  - Lưu trữ và phục hồi trạng thái mô hình an toàn, sẵn sàng xuất file nộp bài cho ban tổ chức.
* **What I still need to practice:**
  - Áp dụng toàn bộ kiến thức PyTorch để bước vào **Module M4: AI nhìn và hiểu dữ liệu (Computer Vision & NLP)**!



## 11. NEXT SESSION — BƯỚC ĐI TIẾP THEO


Xin chúc mừng bạn đã chính thức tốt nghiệp Module M3 và nhận Chứng chỉ **Certificate of Deep Learning Practice**!
Trong buổi tiếp theo (**M4_S01**), chúng ta sẽ bước vào Module thực chiến lớn nhất: **Module M4: AI NHÌN VÀ HIỂU DỮ LIỆU** — Bắt đầu chặng đường đầu tiên: Thị giác máy tính (Computer Vision) với chủ đề Biểu diễn ảnh số, Tensors hình ảnh và các kênh màu Grayscale/RGB!




---

## 🏆 MODULE 3 ASSESSMENT & CERTIFICATION

### Phần A: Đánh giá Lý thuyết & Tư duy Deep Learning (Theory & DL Architecture)
- Nắm vững cơ chế Autograd, Computational Graph và 5 bước của Training Step.
- Phân biệt sâu sắc vai trò của Activation phi tuyến (ReLU), `model.train()` vs `model.eval()`.
- Giải thích bản chất của Dropout, Weight Decay và chiến thuật Checkpointing an toàn.

### Phần B: Sản phẩm Thực hành Bắt buộc (Practical Deliverable)
- **Notebook PyTorch có Training Loop tự viết:** Huấn luyện thành công mạng nơ-ron MLP trên dữ liệu thực tế, có tách bạch pha Train và pha Validation không dùng thư viện bọc ngoài.
- **Model Checkpoint (.pt):** Xuất bản thành công file lưu trữ trọng số `state_dict` tốt nhất kèm code kiểm thử tái nạp (load) không lỗi.

### Tiêu chí Đạt chuẩn (Rubric):
| Tiêu chí | Trọng số | Yêu cầu đạt |
| :--- | :--- | :--- |
| **Data Pipeline (Dataset & Loader)** | 25% | Xây dựng Custom Dataset và DataLoader batching chuẩn |
| **Kiến trúc mạng (nn.Module)** | 25% | Thiết kế mạng MLP có Activation và Dropout hợp lý |
| **Training & Validation Loop** | 30% | Viết đủ 5 bước cập nhật, theo dõi Loss và Accuracy |
| **Checkpointing & Regularization**| 20% | Lưu và nạp thành công `best_model.pt`, có Early Stopping |

**Chứng chỉ hoàn thành:**
🏅 **Certificate of Deep Learning Practice** — Đủ điều kiện tiến vào Module M4 (Computer Vision & NLP Applied AI).


---

## 🎯 Syllabus Traceability (Đối chiếu Syllabus)

Buổi học này đóng góp trực tiếp vào mục tiêu của **M3**:
- Chẩn đoán Overfitting qua Learning Curves ✓
- Kỹ thuật Regularization: Weight decay & Dropout ✓
- Tự viết Training loop & Checkpoint (Save/Load state_dict) ✓
- Nhận chứng chỉ Certificate of Deep Learning Practice ✓

* **Session Deliverable:** Notebook PyTorch có training loop + Checkpoint (.pt) hoàn chỉnh
* **Đảm bảo chuẩn Syllabus Lock:** Nội dung nằm đúng chuẩn 100%, không lan man, giải quyết trực tiếp năng lực cốt lõi.
